In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
sales_df = spark.table("workspace.default.silver_sales")

In [0]:
sales_df.show(10, False)

In [0]:
window_spec = Window.partitionBy("customer_id").orderBy("order_date")

In [0]:
sales_df = sales_df.withColumn(
    "next_order_date",
    lead("order_date", 1).over(window_spec)
)

In [0]:
sales_df.select(
    "customer_id",
    "order_date",
    "next_order_date"
).show(20, False)

In [0]:
sales_df = sales_df.withColumn(
    "previous_order_date",
    lag("order_date", 1).over(window_spec)
)

In [0]:
sales_df.select(
    "customer_id",
    "order_date",
    "previous_order_date"
).show(20, False)

In [0]:
customer_revenue_df = (
    sales_df
    .groupBy("customer_id", "customer_name")
    .agg(
        sum("total_amount").alias("customer_revenue")
    )
)

In [0]:
customer_revenue_df.show()

In [0]:
rank_window = Window.orderBy(col("customer_revenue").desc())

In [0]:
from pyspark.sql.functions import row_number

row_number_df = customer_revenue_df.withColumn(
    "row_num",
    row_number().over(rank_window)
)

In [0]:
row_number_df.show()

In [0]:
from pyspark.sql.functions import rank

rank_df = customer_revenue_df.withColumn(
    "rank",
    rank().over(rank_window)
)

In [0]:
rank_df.show()

In [0]:
from pyspark.sql.functions import dense_rank

dense_rank_df = customer_revenue_df.withColumn(
    "dense_rank",
    dense_rank().over(rank_window)
)

In [0]:
dense_rank_df.show()

In [0]:
source_df = spark.read.table("workspace.default.customer_updates")

In [0]:
target_table = "workspace.default.customers"

In [0]:
from delta.tables import DeltaTable

In [0]:
target = DeltaTable.forName(
    spark,
    target_table
)

In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

In [0]:
from pyspark.sql import Row

updates = [

    Row(customer_id=1001,
        customer_name="Customer 1001 Updated",
        email="customer1001@gmail.com",
        city="Chennai"),

    Row(customer_id=1025,
        customer_name="New Customer",
        email="newcustomer@gmail.com",
        city="Bangalore")

]

updates_df = spark.createDataFrame(updates)

updates_df.show()

In [0]:
updates_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("default.customer_updates")

In [0]:
spark.sql("SELECT * FROM default.customer_updates").show()

In [0]:
from delta.tables import DeltaTable

In [0]:
target = DeltaTable.forName(
    spark,
    "default.silver_customers"
)

In [0]:
source = spark.table("default.customer_updates")

In [0]:
spark.sql("""
SELECT
customer_id,
customer_name,
email
FROM default.silver_customers
WHERE customer_id IN (1001,1025)
""").show(truncate=False)

In [0]:
from pyspark.sql import Row

updates = [

    Row(
        customer_id=1001,
        customer_name="Customer 1001 Updated",
        email="customer1001@gmail.com",
        city="Chennai"
    ),

    Row(
        customer_id=2001,
        customer_name="New Customer",
        email="newcustomer@gmail.com",
        city="Bangalore"
    )

]

updates_df = spark.createDataFrame(updates)

updates_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable("default.customer_updates")

In [0]:
spark.sql("""
SELECT *
FROM default.customer_updates
""").show(truncate=False)

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "default.silver_customers"
)

source = spark.table("default.customer_updates")

target.alias("t").merge(
    source.alias("s"),
    "t.customer_id = s.customer_id"
).whenMatchedUpdate(
    set={
        "customer_name": "s.customer_name",
        "email": "s.email",
        "city": "s.city"
    }
).whenNotMatchedInsert(
    values={
        "customer_id": "s.customer_id",
        "customer_name": "s.customer_name",
        "email": "s.email",
        "city": "s.city"
    }
).execute()

In [0]:
spark.sql("""
SELECT
customer_id,
customer_name,
email,
city
FROM default.silver_customers
WHERE customer_id IN (1001,2001)
ORDER BY customer_id
""").show(truncate=False)